# Projekt 2 - Regresja logistyczna. Walidacja krzyżowa. Techniki ewaluacji modelu.

## 1. Instalacja wymaganych bibliotek

In [ ]:
%pip install kagglehub

## 2. Import bibliotek i przygotowanie środowiska

In [ ]:
from collections import defaultdict
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split

## 3. Pobranie i przygotowanie danych

In [23]:
DATASET_SLUG = "abdallahalidev/plantvillage-dataset"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

dataset_path = Path(kagglehub.dataset_download(DATASET_SLUG))
print(f"Dataset downloaded to: {dataset_path}")

class_to_files = defaultdict(list)
for file_path in dataset_path.rglob("*"):
    if file_path.is_file() and file_path.suffix.lower() in IMAGE_EXTENSIONS:
        class_to_files[file_path.parent.name].append(file_path)

class_to_files = {
    class_name: sorted(paths)
    for class_name, paths in class_to_files.items()
    if len(paths) > 0
}

class_counts = pd.Series(
    {class_name: len(paths) for class_name, paths in class_to_files.items()}
).sort_values(ascending=False)

print(f"Number of classes found: {len(class_counts)}")
display(class_counts.to_frame("num_images"))

Dataset downloaded to: C:\Users\Michal\.cache\kagglehub\datasets\abdallahalidev\plantvillage-dataset\versions\3
Number of classes found: 38


,num_images
Orange___Haunglongbing_(Citrus_greening),16521
Tomato___Tomato_Yellow_Leaf_Curl_Virus,16071
Soybean___healthy,15270
Peach___Bacterial_spot,6891
Tomato___Bacterial_spot,6381
Tomato___Late_blight,5727
Squash___Powdery_mildew,5505
Tomato___Septoria_leaf_spot,5313
Tomato___Spider_mites Two-spotted_spider_mite,5028
Apple___healthy,4935


## 4. Wybór par klas do porównania

In [15]:
PREFERRED_PAIRS = [
    ("Tomato___healthy", "Tomato___Early_blight"),
    ("Potato___healthy", "Potato___Late_blight"),
    ("Apple___healthy", "Apple___Apple_scab"),
]

selected_pairs = []
available_classes = set(class_counts.index)

for c1, c2 in PREFERRED_PAIRS:
    if c1 in available_classes and c2 in available_classes:
        selected_pairs.append((c1, c2))

# 3. Wyświetlamy wybrane pary
print(f"Wybrane pary do analizy: {selected_pairs}")

Wybrane pary do analizy: [('Tomato___healthy', 'Tomato___Early_blight'), ('Potato___healthy', 'Potato___Late_blight'), ('Apple___healthy', 'Apple___Apple_scab')]


## 5. Konfiguracja parametrów i przetwarzanie obrazów

In [17]:
RANDOM_STATE = 42
IMAGE_SIZE = (48, 48)
MAX_IMAGES_PER_CLASS = 300
TEST_SIZE = 0.2


def load_image_as_vector(image_path, image_size):
    with Image.open(image_path) as img:
        img_arr = np.array(img.convert("RGB").resize(image_size), dtype=np.float32)
    return (img_arr / 255.0).flatten()


def build_binary_dataset(
    class_to_files,
    class_pair,
    image_size=IMAGE_SIZE,
    max_per_class=MAX_IMAGES_PER_CLASS,
    random_state=RANDOM_STATE,
):
    X, y = [], []
    rng = np.random.default_rng(random_state)

    limit = min(
        max_per_class,
        len(class_to_files[class_pair[0]]),
        len(class_to_files[class_pair[1]]),
    )

    for label, class_name in enumerate(class_pair):
        files = class_to_files[class_name]
        selected_files = rng.choice(files, size=limit, replace=False)

        for f in selected_files:
            try:
                X.append(load_image_as_vector(f, image_size))
                y.append(label)
            except Exception as e:
                print(f"Błąd pliku {f}: {e}")

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

## 6. Definicja procedury trenowania i ewaluacji modelu

In [24]:
def evaluate_pair(
    class_pair, class_to_files, image_size, max_images, test_size, random_state
):
    X, y = build_binary_dataset(
        class_to_files, class_pair, image_size, max_images, random_state
    )
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )

    model = LogisticRegression(
        solver="saga", max_iter=2000, random_state=random_state, n_jobs=-1
    )
    scoring = ["accuracy", "precision", "recall", "f1"]

    cv_results = cross_validate(
        model, X_train, y_train, cv=5, scoring=scoring, n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    return {
        "pair": f"{class_pair[0]} vs {class_pair[1]}",
        "labels": class_pair,
        "sample_count": len(y),
        "test": {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
        },
        "cv_mean": {m: cv_results[f"test_{m}"].mean() for m in scoring},
        "confusion_matrix": confusion_matrix(y_test, y_pred),
    }

## 7. Uruchomienie oceny dla wybranych par klas

In [20]:
pair_results = []

for pair in selected_pairs:
    result = evaluate_pair(
        pair, class_to_files, IMAGE_SIZE, MAX_IMAGES_PER_CLASS, TEST_SIZE, RANDOM_STATE
    )
    pair_results.append(result)
    print(f"Ukończono analizę: {result['pair']}")

c:\Users\Michal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Ukończono analizę: Tomato___healthy vs Tomato___Early_blight


c:\Users\Michal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Ukończono analizę: Potato___healthy vs Potato___Late_blight


c:\Users\Michal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Ukończono analizę: Apple___healthy vs Apple___Apple_scab


## 8. Zestawienie wyników w tabeli podsumowującej

In [22]:
rows = []
for result in pair_results:
    rows.append(
        {
            "Para klas": result["pair"],
            "Accuracy": result["test"]["accuracy"],
            "Precision": result["test"]["precision"],
            "Recall": result["test"]["recall"],
            "F1-Score": result["test"]["f1"],
        }
    )

summary_df = pd.DataFrame(rows).sort_values(by="F1-Score", ascending=False)
summary_df

,Para klas,Accuracy,Precision,Recall,F1-Score
1,Potato___healthy vs Potato___Late_blight,0.850000,0.862069,0.833333,0.847458
0,Tomato___healthy vs Tomato___Early_blight,0.841667,0.901961,0.766667,0.828829
2,Apple___healthy vs Apple___Apple_scab,0.800000,0.821429,0.766667,0.793103


## Podsumowanie

### 1. Dane
* **Zbiór danych**: *PlantVillage* (obrazy liści roślin zdrowych i chorych).
* **Wybrane pary**: 
    1. Tomato (Healthy vs Early Blight)
    2. Potato (Healthy vs Late Blight)
    3. Apple (Healthy vs Apple Scab)
* **Liczebność**: Po **300 obrazów** na każdą klasę (łącznie 600 na parę).
* **Reprezentacja**: Obrazy RGB przeskalowane do formatu **48x48 pikseli** i znormalizowane do przedziału [0, 1].

### 2. Podział i Walidacja
* **Podział danych**: **80%** zbiór treningowy, **20%** zbiór testowy.
* **Walidacja krzyżowa**: Zastosowano 5 'foldów'.
* **Model**: Regresja logistyczna (algorytm `SAGA`), która uczy się liniowej granicy decyzyjnej między wektorami cech obrazu.

### 3. Wyniki i Wnioski
* **Skuteczność**: Model uzyskał wyniki **F1-Score w granicach 79-85%**.
* **Najłatwiejsza para**: **Ziemniak (Potato)** – wysokie wyniki sugerują, że zaraza ziemniaczana daje bardzo charakterystyczne zmiany wizualne.
* **Najtrudniejsza para**: **Jabłko (Apple)** – niższy F1-Score wskazuje na większe podobieństwo wizualne parchu do zdrowej tkanki przy niskiej rozdzielczości obrazu.
* **Ewaluacja**: Wykorzystanie Precision i Recall pozwoliło wykazać, że model nie tylko ma wysoką ogólną dokładność, ale też skutecznie balansuje między wykrywaniem chorób a unikaniem fałszywych alarmów.